# Day 075 — Exercise 2: image_to_frames

**What you'll build:** `image_to_frames(image, n_frames, capture_fn=None) -> list` — convert a face image into a list of identical BGR numpy frame arrays.

**Why it matters:** Stage 2 of the pipeline — turns a static face photo into the frame sequence that VideoProcessor writes to a video file.

In [ ]:
from pathlib import Path
import tempfile

def _make_mock_frames(n=5, height=32, width=32):
    import numpy as np
    return [np.zeros((height, width, 3), dtype=np.uint8) for _ in range(n)]
_mock_capture_fn = lambda img, n: _make_mock_frames(n)


## Task

- **Mock:** `return capture_fn(image, n_frames)`
- **Real:** lazy-import `numpy` and `PIL`; if `image` is not a PIL Image, open and convert to `'RGB'`; `arr = np.array(image)[:, :, ::-1].astype(np.uint8)` (RGB → BGR); return `[arr.copy() for _ in range(n_frames)]`

**Key:** `arr.copy()` per frame — each frame must be an independent array.

## Your Implementation

In [ ]:
def image_to_frames(image, n_frames: int, capture_fn=None) -> list:
    """Create n_frames identical BGR frames from a face image.

    Args:
        image:      PIL Image or path to image file
        n_frames:   number of frames to generate
        capture_fn: callable(image, n_frames) -> list[np.ndarray] for testing
    Returns:
        list of (H, W, 3) uint8 BGR numpy arrays, n_frames elements
    """
    raise NotImplementedError


In [ ]:
def image_to_frames(image, n_frames, capture_fn=None):
    if capture_fn is not None:
        return capture_fn(image, n_frames)
    import numpy as np
    from PIL import Image as PILImage
    if not isinstance(image, PILImage.Image):
        image = PILImage.open(str(image)).convert('RGB')
    arr = np.array(image)[:, :, ::-1].astype(np.uint8)
    return [arr.copy() for _ in range(n_frames)]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # returns a list
    frames = image_to_frames('face.png', 5, capture_fn=_mock_capture_fn)
    assert isinstance(frames, list)
    score += 1; print("✅ returns a list")

    # correct number of frames
    assert len(frames) == 5, f"expected 5 frames, got {len(frames)}"
    score += 1; print("✅ correct frame count (5)")

    # frames are (H, W, 3) uint8 numpy arrays
    assert frames[0].shape == (32, 32, 3)
    assert frames[0].dtype.name == 'uint8'
    score += 1; print("✅ frames are (32, 32, 3) uint8 numpy arrays")

    # capture_fn receives both image and n_frames
    captured = {}
    def _cap(img, n):
        captured['n'] = n
        return _make_mock_frames(n)
    image_to_frames('face.png', 7, capture_fn=_cap)
    assert captured.get('n') == 7
    score += 1; print("✅ capture_fn receives n_frames correctly")

    # different n_frames → different list lengths
    f3 = image_to_frames('face.png', 3, capture_fn=_mock_capture_fn)
    f8 = image_to_frames('face.png', 8, capture_fn=_mock_capture_fn)
    assert len(f3) == 3 and len(f8) == 8
    score += 1; print("✅ n_frames controls output list length")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def image_to_frames(image, n_frames, capture_fn=None):
    if capture_fn is not None:
        return capture_fn(image, n_frames)
    import numpy as np
    from PIL import Image as PILImage
    if not isinstance(image, PILImage.Image):
        image = PILImage.open(str(image)).convert('RGB')
    arr = np.array(image)[:, :, ::-1].astype(np.uint8)
    return [arr.copy() for _ in range(n_frames)]
```

**Why `[:, :, ::-1]`?** PIL produces RGB; OpenCV/VideoWriter expects BGR. Reversing the channel axis (axis 2) swaps R and B while keeping G. It is a zero-copy numpy view — no data is duplicated for the conversion.

</details>